# Crop a 4D-STEM scan to the sample

Type the path of a reshaped 4D-STEM HDF5 file, **or of a folder**, into the tool and press Enter.
With a folder, every HDF5 file in it and in all its subfolders is lined up and shown one at a time.
For each file you get a virtual image (ADF / BF / total sum) with a box you can drag over the sample.
**Crop & save** writes only that part of the scan to a new file and moves on to the next one.

**Setup (once):** `pip install h5py ipympl isal` and restart the kernel (`isal` is optional: it makes
gzip decompression about 1.5-2x faster). This notebook uses the interactive
`%matplotlib widget` backend, which works in JupyterLab, Notebook 7 and VS Code.

**Use:** run all cells (*Run → Run All Cells*), then type or paste a file or folder path into the tool in section 4.

**Nothing is reoriented.** The first two axes of the 4D dataset are the scan and the last two the
detector, in the order they are stored. The image is shown exactly as stored (axis 0 down, axis 1
across, labelled with the names in the dataset's `axes` attribute, e.g. `scan_x, scan_y, ky, kx`).
The cropped file is a plain slice `data[a0:b0, a1:b1, :, :]` with the same axis order, dtype,
chunking, compression and attributes, and every saved file is read back and checked against the
original. Files are read in blocks, so a dataset doesn't have to fit in memory.

In [1]:
%matplotlib widget
import csv, fnmatch, functools, re, time, traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import h5py
try:
    import hdf5plugin  # extra HDF5 compression filters (bitshuffle, LZ4, ...), if installed
except ImportError:
    pass
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, PowerNorm
from matplotlib.figure import Figure
from matplotlib.patches import Circle, Rectangle
from matplotlib.widgets import RectangleSelector
import ipywidgets as widgets
from IPython.display import display

## 1. Settings

In [2]:
PATH = r"/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_115357/20260524_115357_data.hdf5"              # optional: a file, or a folder (all HDF5 files in it and its subfolders), to open
                        # straight away. You can also just type paths into the tool.
NAME_FILTER = None        # only files whose name matches, e.g. "*reshaped*"; "" = every .h5/.hdf5/.hdf/.emd
OUT_FOLDER = r"/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/Zarr/Reshapped/cropped_85,85"        # "" -> a "cropped" folder next to each input file;
                        # otherwise everything goes here, keeping the subfolder layout
DATASET = None  
         # path of the 4D data inside the files, e.g. "/data"; None = the largest 4D dataset

BOX = None              # starting ROI size in scan pixels: 64 or (rows, cols); None = 1/3 of the scan
LOCK_SIZE = False       # start with the ROI size locked (drawing then snaps to that size)
SKIP_DONE = True        # when moving on, skip files that already have a crop (lets you stop and resume)
PREFETCH = True         # read the next file in the background while you draw the ROI
WORKERS = None          # processes that read files in parallel; None = up to 8 (one fewer than your CPUs), 0 = off
COPY_METADATA = True    # also copy the file's other groups/attributes into the cropped file
UPDATE_SIZE_ATTRS = True  # set scan_width / scan_height attributes (if present) to the cropped size,
                          # matched to the axes by the dataset's own 'axes' names; originals are kept
                          # as crop_original_scan_width / crop_original_scan_height

# Starting virtual detectors in detector pixels. None = found automatically from the mean diffraction
# pattern. You can also change them in the tool (sliders under the pattern, or drag the circles).
CENTER = None           # (cy, cx)
BF_RADIUS = None
ADF_INNER = None        # default: 1.2 x BF radius
ADF_OUTER = None        # default: edge of the detector

# Starting display contrast of the virtual image (only changes what you see, never the data).
CONTRAST = (0.5, 99.5)  # low / high percentile of the image mapped to black / white
GAMMA = 1.0             # < 1 brightens faint areas, > 1 darkens them

## 2. Helper functions
Run this cell once; nothing here needs editing.

In [3]:
BLOCK_BYTES = 256e6      # data (as float32) read per block; lower it if memory is tight
VIEWS = ("ADF", "BF", "Sum")
H5_SUFFIXES = (".h5", ".hdf5", ".hdf", ".emd")


# ---- reading / writing HDF5 -------------------------------------------------
def clean_path(text):
    """Path from typed text; copes with surrounding quotes (e.g. Windows 'Copy as path')."""
    return Path(text.strip().strip('"').strip("'")).expanduser()


def natural_key(p):
    """Sort 'folder 2' before 'folder 10'."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(p))]


def find_h5_files(root, name_filter=""):
    """All HDF5 files in `root` and its subfolders, leaving out this tool's own *_crop outputs.
    name_filter: optional wildcard(s) on the file name, e.g. "*reshaped*" or "*.h5, *.emd"."""
    root = Path(root)
    pats = [s.strip().lower() for s in name_filter.split(",") if s.strip()]
    found = []
    for p in root.rglob("*"):
        rel = p.relative_to(root)
        if (p.suffix.lower() in H5_SUFFIXES and not p.stem.endswith("_crop") and p.is_file()
                and not any(part.startswith(".") for part in rel.parts)
                and (not pats or any(fnmatch.fnmatch(p.name.lower(), pat) for pat in pats))):
            found.append(p)
    return sorted(found, key=natural_key)


def find_4d_dataset(f, path=None):
    """The 4D dataset at `path`, or the largest 4D dataset in the file."""
    if path:
        d = f[path]
        if not isinstance(d, h5py.Dataset) or d.ndim != 4:
            raise ValueError(f"{path} is not a 4D dataset")
        return d
    found = []
    f.visititems(lambda name, obj: found.append(obj)
                 if isinstance(obj, h5py.Dataset) and obj.ndim == 4 else None)
    if not found:
        raise ValueError("no 4D dataset in this file (set DATASET to its path)")
    return max(found, key=lambda d: d.size)


def rows_per_block(dset):
    """Scan rows per read so that one block is about BLOCK_BYTES."""
    ny, nx, ky, kx = dset.shape
    n = int(max(1, min(ny, BLOCK_BYTES // (nx * ky * kx * 4))))
    if dset.chunks and dset.chunks[0] > 1:          # align with the HDF5 chunks
        c = dset.chunks[0]
        n = max(c, (n // c) * c)
    return min(n, ny)


def copy_attrs(src, dst):
    for k, v in src.attrs.items():
        try:
            dst.attrs[k] = v
        except Exception:      # e.g. object references
            pass


def copy_everything_except(src_grp, dst_grp, skip, scan_shape, notes):
    """Copy every object in src_grp to dst_grp except the dataset at path `skip`."""
    for name in src_grp:
        try:
            obj = src_grp[name]
        except KeyError:        # broken link
            continue
        if obj.name == skip:
            continue
        if isinstance(obj, h5py.Group) and skip.startswith(obj.name.rstrip("/") + "/"):
            g = dst_grp.require_group(name)
            copy_attrs(obj, g)
            copy_everything_except(obj, g, skip, scan_shape, notes)
        else:
            src_grp.copy(obj, dst_grp, name=name)
            if isinstance(obj, h5py.Dataset) and obj.shape[:2] == tuple(scan_shape):
                notes.append(f"{obj.name} {obj.shape} has the scan shape and was copied uncropped")


def axis_names(dset):
    """The dataset's own names for its 4 axes (from an 'axes' attribute), else axis 0..3."""
    names = ["axis 0", "axis 1", "axis 2", "axis 3"]
    a = dset.attrs.get("axes")
    if a is not None:
        if isinstance(a, np.ndarray):
            a = ", ".join(x.decode() if isinstance(x, bytes) else str(x) for x in a.ravel())
        elif isinstance(a, bytes):
            a = a.decode()
        parts = [s.strip() for s in str(a).replace(";", ",").split(",") if s.strip()]
        if len(parts) == 4:
            names = parts
    return names


def new_size_attrs(attrs, names, new_scan_shape):
    """New scan_width / scan_height values, matched to axes by the dataset's own axis names."""
    size = {}
    for i in (0, 1):
        n = names[i].lower()
        if n.endswith("x"):
            size["scan_width"] = new_scan_shape[i]
        elif n.endswith("y"):
            size["scan_height"] = new_scan_shape[i]
    if len(size) < 2 and new_scan_shape[0] == new_scan_shape[1]:     # square: no ambiguity
        size = {"scan_width": new_scan_shape[0], "scan_height": new_scan_shape[0]}
    return {k: v for k, v in size.items() if k in attrs}


def one_frame_chunks(d):
    """True if every HDF5 chunk holds exactly one diffraction pattern."""
    return d.chunks is not None and tuple(d.chunks[:2]) == (1, 1) and tuple(d.chunks[2:]) == tuple(d.shape[2:])


def create_like(group, name, src, new_shape):
    """New dataset with the source's own storage settings: same chunking and the same compression
    filters, whatever they are (gzip, bitshuffle/LZ4, Blosc, ...), copied at the HDF5 level."""
    dcpl = src.id.get_create_plist()                     # a copy of the source's creation settings
    layout = dcpl.get_layout()
    if layout == h5py.h5d.CHUNKED:
        chunks = tuple(min(c, s) for c, s in zip(src.chunks, new_shape))
        if chunks != tuple(src.chunks):
            dcpl.set_chunk(chunks)
    elif layout != h5py.h5d.CONTIGUOUS:                  # virtual / compact source: plain chunked copy
        dcpl = h5py.h5p.create(h5py.h5p.DATASET_CREATE)
        dcpl.set_chunk((1, 1) + tuple(new_shape[2:]))
    else:
        dcpl = h5py.h5p.create(h5py.h5p.DATASET_CREATE)  # uncompressed, contiguous like the source
    def make():
        return h5py.Dataset(h5py.h5d.create(group.id, name.encode(), src.id.get_type(),
                                            h5py.h5s.create_simple(new_shape), dcpl=dcpl))
    dst = make()
    if layout == h5py.h5d.CHUNKED:
        # Some filters (e.g. bitshuffle) add values in front of their settings every time a dataset is
        # created, so copying finished settings would add them twice. Strip them and create again, so
        # the crop's compression settings come out identical to the source's.
        want, got = filter_pipeline(src), filter_pipeline(dst)
        if want != got and len(want) == len(got):
            new, fixed = [], False
            for (code, flags, wv), (code2, _, gv) in zip(want, got):
                k = len(gv) - len(wv)
                if code == code2 and k > 0 and tuple(gv[k:]) == tuple(wv):
                    wv, fixed = tuple(wv[k:]), True
                new.append((code, flags, tuple(wv)))
            if fixed:                                    # rebuild the filter list in the same order
                dcpl.remove_filter(h5py.h5z.FILTER_ALL)
                for code, flags, vals in new:
                    dcpl.set_filter(code, flags, vals)
                del group[name]
                dst = make()
    return dst


def write_crop(src_path, dset_path, roi, out_path, copy_metadata=True, update_size_attrs=True, progress=None):
    """Write data[a0:b0, a1:b1, :, :] with the same axis order, dtype, chunks, compression and attributes."""
    y0, y1, x0, x1 = roi                       # y = axis 0, x = axis 1 (whatever the file calls them)
    notes = []
    tmp = out_path.with_name(out_path.name + ".part")
    try:
        with h5py.File(src_path, "r") as fs, h5py.File(tmp, "w") as fd:
            src = fs[dset_path]
            if copy_metadata:
                copy_attrs(fs, fd)
                copy_everything_except(fs, fd, src.name, src.shape[:2], notes)
            new_shape = (y1 - y0, x1 - x0) + src.shape[2:]
            parent, name = src.name.rsplit("/", 1)
            dst = create_like(fd.require_group(parent or "/"), name, src, new_shape)
            copy_attrs(src, dst)
            names = axis_names(src)
            dst.attrs["crop_source_file"] = Path(src_path).name
            dst.attrs["crop_original_shape"] = np.array(src.shape)
            dst.attrs["crop_axis0_range"] = np.array([y0, y1])           # along names[0]
            dst.attrs["crop_axis1_range"] = np.array([x0, x1])           # along names[1]
            dst.attrs["crop_note"] = (f"plain slice data[{y0}:{y1}, {x0}:{x1}, :, :] of the source "
                                      f"({names[0]}, {names[1]}); no axes swapped, flipped or rotated")
            if update_size_attrs:
                for k, v in new_size_attrs(src.attrs, names, new_shape[:2]).items():
                    old = src.attrs[k]
                    dst.attrs[f"crop_original_{k}"] = old
                    dst.attrs[k] = np.asarray(v).astype(np.asarray(old).dtype)
                    notes.append(f"attribute {k}: {old} -> {v}")
            if one_frame_chunks(src) and dst.chunks == src.chunks and filter_pipeline(src) == filter_pipeline(dst):
                # one pattern per chunk and identical compression settings: copy the compressed chunks
                # byte for byte (no decompress / recompress, so this is fast and exact by construction)
                zero = (0,) * (src.ndim - 2)
                for i in range(y0, y1):
                    for j in range(x0, x1):
                        try:
                            mask, raw = src.id.read_direct_chunk((i, j) + zero)
                        except Exception:            # chunk never written in the source
                            dst[i - y0, j - x0] = src[i, j]
                            continue
                        dst.id.write_direct_chunk((i - y0, j - x0) + zero, raw, filter_mask=mask)
                    if progress:
                        progress((i + 1 - y0) / (y1 - y0), f"writing {out_path.name}: {i + 1 - y0}/{y1 - y0} rows")
            else:
                step = rows_per_block(src)
                for r0 in range(y0, y1, step):
                    r1 = min(y1, r0 + step)
                    dst[r0 - y0:r1 - y0] = src[r0:r1, x0:x1]
                    if progress:
                        progress((r1 - y0) / (y1 - y0), f"writing {out_path.name}: {r1 - y0}/{y1 - y0} rows")
    except BaseException:
        tmp.unlink(missing_ok=True)          # don't leave a half-written file behind
        raise
    tmp.replace(out_path)
    return notes


# ---- virtual images ---------------------------------------------------------
def estimate_mean_dp(dset, n_side=32):
    """Mean diffraction pattern from a sparse grid of about n_side x n_side scan positions."""
    ny, nx = dset.shape[:2]
    sy, sx = max(1, ny // n_side), max(1, nx // n_side)
    acc, n = np.zeros(dset.shape[2:], np.float64), 0
    layout = W.chunk_layout(dset)
    for iy in range(0, ny, sy):           # frame by frame: a strided HDF5 read of compressed chunks is very slow
        for ix in range(0, nx, sx):
            acc += W.read_frame(dset, iy, ix, layout)
            n += 1
    return acc / n


def detect_bf_disk(dp):
    """Centre (cy, cx) and radius of the bright-field disk."""
    dp = np.asarray(dp, np.float64)
    hi, lo = np.percentile(dp, 99.9), np.median(dp)
    disk = dp > lo + 0.5 * (hi - lo)
    if disk.sum() < 3:
        return (dp.shape[0] - 1) / 2, (dp.shape[1] - 1) / 2, min(dp.shape) / 8
    yy, xx = np.nonzero(disk)
    return yy.mean(), xx.mean(), np.sqrt(disk.sum() / np.pi)


def detector_geometry(dset, center=None, bf_radius=None, adf_inner=None, adf_outer=None, mean_dp=None):
    cy, cx, r = detect_bf_disk(estimate_mean_dp(dset) if mean_dp is None else mean_dp)
    if center is not None:
        cy, cx = center
    if bf_radius is not None:
        r = bf_radius
    return dict(cy=cy, cx=cx, r_bf=r,
                r_in=adf_inner if adf_inner is not None else 1.2 * r,
                r_out=adf_outer if adf_outer is not None else np.inf)


def robust_limits(img, lo=0.5, hi=99.5):
    vals = img[np.isfinite(img)]
    if vals.size == 0:
        return 0, 1
    vmin, vmax = np.percentile(vals, [lo, hi])
    return (vmin, vmax) if vmax > vmin else (vals.min(), vals.max() + 1e-12)


def display_norm(img, contrast=(0.5, 99.5), gamma=1.0):
    """Colour scaling for showing an image: percentiles `contrast` -> black / white, then gamma."""
    vmin, vmax = robust_limits(img, *contrast)
    if abs(gamma - 1) < 1e-3:
        return Normalize(vmin, vmax)
    return PowerNorm(gamma, vmin, vmax)


def same_geom(a, b, tol=0.06):
    """True if two detector geometries agree to within `tol` pixels (slider readouts show 0.1 px)."""
    for k in ("cy", "cx", "r_bf", "r_in", "r_out"):
        x, y = float(a[k]), float(b[k])
        if np.isfinite(x) != np.isfinite(y) or (np.isfinite(x) and abs(x - y) > tol):
            return False
    return True


def fmt_r(r):
    return f"{r:.1f}" if np.isfinite(r) else "edge"


# ---- ROI helpers: roi = (y0, y1, x0, x1), half-open pixel indices -------------
def clamp_roi(roi, shape):
    ny, nx = shape
    y0, y1, x0, x1 = (int(v) for v in roi)
    y0, x0 = min(max(y0, 0), ny - 1), min(max(x0, 0), nx - 1)
    return y0, min(max(y1, y0 + 1), ny), x0, min(max(x1, x0 + 1), nx)


def box_at(cy, cx, size, shape):
    """Box of `size` (rows, cols) centred on (cy, cx) in pixel-edge coordinates, kept inside the scan."""
    ny, nx = shape
    h, w = min(size[0], ny), min(size[1], nx)
    y0 = min(max(int(np.floor(cy - h / 2 + 0.5)), 0), ny - h)
    x0 = min(max(int(np.floor(cx - w / 2 + 0.5)), 0), nx - w)
    return y0, y0 + h, x0, x0 + w


def normalise_box(box):
    if box is None:
        return None
    box = [box] if np.isscalar(box) else list(box)
    box = box * 2 if len(box) == 1 else box
    if len(box) != 2 or min(box) < 1:
        raise ValueError("BOX must be a positive integer or (rows, cols)")
    return int(box[0]), int(box[1])


# ---- outputs ------------------------------------------------------------------
def verify_crop(src_path, dset_path, roi, out_path, sum_original, workers=None, progress=None):
    """Read the saved file back and check it is exactly data[a0:b0, a1:b1, :, :] of the source:
    same shape/axis order/dtype, sampled frames bit-identical, total-sum image identical."""
    y0, y1, x0, x1 = roi
    with h5py.File(src_path, "r") as fs, h5py.File(out_path, "r") as fo:
        s, o = fs[dset_path], fo[dset_path]
        expect = (y1 - y0, x1 - x0) + s.shape[2:]
        if o.shape != expect or o.dtype != s.dtype:
            raise RuntimeError(f"check failed: saved {o.shape} {o.dtype}, expected {expect} {s.dtype}")
        h, w = expect[:2]
        rng = np.random.default_rng(0)
        pts = {(0, 0), (0, w - 1), (h - 1, 0), (h - 1, w - 1), (h // 2, w // 2)}
        pts |= set(zip(rng.integers(0, h, 20).tolist(), rng.integers(0, w, 20).tolist()))
        for a, b in sorted(pts):
            if not np.array_equal(o[a, b], s[y0 + a, x0 + b]):
                raise RuntimeError(f"check failed: saved frame ({a}, {b}) != original ({y0 + a}, {x0 + b})")
        itemsize, dtype = o.dtype.itemsize, o.dtype
    try:
        back = parallel_virtual(out_path, dset_path, expect, itemsize, None, get_pool(workers), progress,
                                total_only=True)["Sum"]
    except BrokenProcessPool:
        reset_pool(failed=True)
        back = parallel_virtual(out_path, dset_path, expect, itemsize, None, None, progress, total_only=True)["Sum"]
    if not np.allclose(back, sum_original[y0:y1, x0:x1], rtol=1e-12, atol=0):
        raise RuntimeError("check failed: total-sum image of the saved file differs from the original")
    return back


def save_preview(img, roi, back, path, title, names, contrast=(0.5, 99.5), gamma=1.0):
    """Left: the original as stored with the ROI. Right: the saved file read back, same orientation.
    Both use the display contrast / gamma set in the tool."""
    y0, y1, x0, x1 = roi
    fig = Figure(figsize=(11, 5.4))
    a, b = fig.subplots(1, 2)
    a.imshow(img, cmap="gray", interpolation="nearest", norm=display_norm(img, contrast, gamma))
    a.add_patch(Rectangle((x0 - 0.5, y0 - 0.5), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=1.5))
    a.set_title(f"original: {title}\n{names[0]} {y0}:{y1}, {names[1]} {x0}:{x1} ({y1 - y0} x {x1 - x0})",
                fontsize=9)
    b.imshow(back, cmap="gray", interpolation="nearest", norm=display_norm(back, contrast, gamma),
             extent=(x0 - 0.5, x1 - 0.5, y1 - 0.5, y0 - 0.5))       # plotted in original coordinates
    b.set_title("saved file, read back (total-sum image)\nidentical to the original inside the box", fontsize=9)
    for ax in (a, b):
        ax.set_ylabel(f"axis 0: {names[0]}")
        ax.set_xlabel(f"axis 1: {names[1]}")
    fig.suptitle("Shown exactly as stored (axis 0 down, axis 1 across); no axes swapped, flipped or rotated",
                 fontsize=10)
    fig.savefig(path, dpi=110, bbox_inches="tight")


def append_log(log_path, row):
    new = not log_path.exists()
    with open(log_path, "a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(row))
        if new:
            w.writeheader()
        w.writerow(row)

# ---- fast parallel reading ----------------------------------------------------------------------
# Worker processes read the compressed chunks straight from the file and decompress them in
# parallel (HDF5 itself only uses one core). The worker code lives in a small module written to the
# temp folder, so it also works on Windows/macOS, where worker processes must import their code.
WORKER_SOURCE = r"""
# Written by crop_4dstem.ipynb so its worker processes can import it. Safe to delete.
import os
for _v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "BLIS_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"          # set before numpy loads: each worker needs one thread, not one per core
import zlib
import numpy as np
import h5py
try:
    import hdf5plugin             # so workers can read bitshuffle / LZ4 / Blosc / Zstd compressed data
except ImportError:
    pass
try:
    from isal import isal_zlib as _zlib        # faster gzip decompression, if installed
except ImportError:
    _zlib = zlib


def open_file(path):
    try:
        return h5py.File(path, "r", locking=False)
    except TypeError:                           # older h5py
        return h5py.File(path, "r")


def chunk_layout(d):
    # (True, position of the deflate filter or None) if every chunk is exactly one diffraction
    # pattern stored uncompressed or with plain gzip (shuffle on 1-byte data changes nothing)
    if d.chunks is None or tuple(d.chunks[:2]) != (1, 1) or tuple(d.chunks[2:]) != tuple(d.shape[2:]):
        return False, None
    pl = d.id.get_create_plist()
    ids = [pl.get_filter(i)[0] for i in range(pl.get_nfilters())]
    if ids == []:
        return True, None
    if ids == [1]:
        return True, 0
    if ids == [2, 1] and d.dtype.itemsize == 1:
        return True, 1
    return False, None


def ignore_sigint():
    # "Interrupt kernel" in Jupyter should stop the notebook, not print tracebacks from every worker
    import signal
    signal.signal(signal.SIGINT, signal.SIG_IGN)


def read_frame(d, i, j, layout=None):
    # One diffraction pattern: straight from its compressed chunk when possible, else through HDF5
    ok, deflate = chunk_layout(d) if layout is None else layout
    if ok:
        try:
            mask, raw = d.id.read_direct_chunk((i, j) + (0,) * (d.ndim - 2))
        except Exception:                       # chunk never written: HDF5 returns the fill value
            return d[i, j]
        if deflate is not None and not (mask >> deflate) & 1:
            raw = _zlib.decompress(raw)
        return np.frombuffer(raw, d.dtype).reshape(d.shape[2:])
    return d[i, j]


def read_rows(d, r0, r1):
    layout = chunk_layout(d)
    if not layout[0]:
        return d[r0:r1]                         # any other layout: let HDF5 do it
    out = np.empty((r1 - r0,) + tuple(d.shape[1:]), d.dtype)
    for i in range(r0, r1):
        for j in range(d.shape[1]):
            out[i - r0, j] = read_frame(d, i, j, layout)
    return out


def virtual_rows(path, dset_path, r0, r1, iv_bf, iv_in, iv_out):
    # Total, BF-disk and ADF-annulus sums for scan rows r0:r1 (exact integer sums for integer data).
    with open_file(path) as f:
        a = read_rows(f[dset_path], r0, r1)
    rows, nx = a.shape[:2]
    a = a.reshape((rows * nx,) + a.shape[2:])
    acc = np.int64 if a.dtype.kind in "ui" else np.float64
    total = a.reshape(a.shape[0], -1).sum(axis=1, dtype=acc)

    def disk(iv):
        s = np.zeros(a.shape[0], acc)
        for y, c0, c1 in iv:
            s += a[:, y, c0:c1].sum(axis=1, dtype=acc)
        return s

    bf = None if iv_bf is None else disk(iv_bf).reshape(rows, nx)
    adf = None
    if iv_in is not None:
        adf = ((total if iv_out is None else disk(iv_out)) - disk(iv_in)).reshape(rows, nx)
    return r0, total.reshape(rows, nx), bf, adf
"""

import importlib, multiprocessing, os, sys, tempfile
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures.process import BrokenProcessPool

_wdir = Path(tempfile.gettempdir()) / "crop4dstem"
_wdir.mkdir(exist_ok=True)
(_wdir / "crop4dstem_worker.py").write_text(WORKER_SOURCE)
if str(_wdir) not in sys.path:
    sys.path.insert(0, str(_wdir))
import crop4dstem_worker as W
W = importlib.reload(W)

if globals().get("_POOL") is not None:          # re-running this cell: start fresh workers
    _POOL.shutdown(wait=False, cancel_futures=True)
_POOL, _POOL_N, _POOL_FAILED = None, 0, False
MAX_AUTO_WORKERS = 8                             # more rarely helps: the disk becomes the limit


def available_cpus():
    try:
        return len(os.sched_getaffinity(0))      # CPUs this process may use (Linux)
    except AttributeError:
        return os.cpu_count() or 2


def get_pool(workers):
    """Shared pool of worker processes (None = run everything in this process)."""
    global _POOL, _POOL_N
    if _POOL_FAILED:
        return None
    n = min(MAX_AUTO_WORKERS, max(1, available_cpus() - 1)) if workers is None else int(workers)
    if n < 1:
        return None
    if _POOL is None or _POOL_N != n:
        if _POOL is not None:
            _POOL.shutdown(wait=False, cancel_futures=True)
        _POOL = ProcessPoolExecutor(n, mp_context=multiprocessing.get_context("spawn"), initializer=W.ignore_sigint)
        _POOL_N = n
    return _POOL


def reset_pool(failed=False):
    """Shut the workers down; with failed=True, don't try them again in this session."""
    global _POOL, _POOL_FAILED
    if _POOL is not None:
        _POOL.shutdown(wait=False, cancel_futures=True)
    _POOL = None
    if failed and not _POOL_FAILED:
        _POOL_FAILED = True
        print("Worker processes could not run, so files are read in this process from now on "
              "(still the fast method). Set WORKERS = 0 to skip trying them.")


def disk_intervals(det_shape, cy, cx, r):
    """Detector pixels with distance <= r from (cy, cx), as (row, col_start, col_stop) runs."""
    if r is None or not np.isfinite(r):
        return None
    out = []
    for y in range(det_shape[0]):
        h2 = r * r - (y - cy) ** 2
        if h2 < 0:
            continue
        dx = np.sqrt(h2)
        c0, c1 = max(int(np.ceil(cx - dx)), 0), min(int(np.floor(cx + dx)) + 1, det_shape[1])
        if c1 > c0:
            out.append((y, c0, c1))
    return out


def parallel_virtual(path, dset_path, shape, itemsize, geom, pool, progress=None, total_only=False):
    ny, nx = shape[:2]
    det = tuple(shape[2:])
    if total_only:
        ivs = (None, None, None)
    else:
        ivs = tuple(disk_intervals(det, geom["cy"], geom["cx"], geom[k]) for k in ("r_bf", "r_in", "r_out"))
    rows = max(1, min(ny, int(64e6 // (nx * int(np.prod(det)) * itemsize))))   # ~64 MB per task
    tasks = [(str(path), dset_path, r0, min(ny, r0 + rows)) + ivs for r0 in range(0, ny, rows)]
    if pool is None:                                       # same fast code, in this process
        futs, results = [], (W.virtual_rows(*t) for t in tasks)
    else:
        futs = [pool.submit(W.virtual_rows, *t) for t in tasks]
        results = (fut.result() for fut in as_completed(futs))
    imgs, done, t0 = {}, 0, time.time()
    try:
        for r0, total, bf, adf in results:
            for name, arr in (("Sum", total), ("BF", bf), ("ADF", adf)):
                if arr is not None:
                    imgs.setdefault(name, np.zeros((ny, nx), arr.dtype))[r0:r0 + arr.shape[0]] = arr
            done += total.shape[0]
            if progress:
                progress(done / ny, f"{'checking' if total_only else 'virtual images'}: {done}/{ny} rows "
                                    f"({time.time() - t0:.0f} s, {_POOL_N if pool else 1} process(es))")
    except BaseException:
        for fut in futs:
            fut.cancel()
        raise
    if not total_only and not ivs[0]:
        imgs.pop("BF", None)
    return {k: imgs[k] for k in VIEWS if k in imgs}


def virtual_images(path, dset_path, geom, workers=None, progress=None):
    """ADF / BF / Sum images of a 4D dataset: in parallel worker processes, else in this process."""
    with h5py.File(path, "r") as f:
        d = f[dset_path]
        shape, itemsize = d.shape, d.dtype.itemsize
    try:
        return parallel_virtual(path, dset_path, shape, itemsize, geom, get_pool(workers), progress)
    except BrokenProcessPool:                       # workers died (e.g. out of memory): one process
        reset_pool(failed=True)
        return parallel_virtual(path, dset_path, shape, itemsize, geom, None, progress)


def filter_pipeline(d):
    pl = d.id.get_create_plist()
    return [tuple(pl.get_filter(i)[:3]) for i in range(pl.get_nfilters())]

## 3. Interactive cropping tool
Run this cell once to define the tool; it is started in the next cell.

In [4]:
def _safe(method):
    """Show errors from button/mouse callbacks in the log box instead of losing them."""
    @functools.wraps(method)
    def wrapper(self, *args, **kwargs):
        try:
            return method(self, *args, **kwargs)
        except Exception:
            self._set_busy(False, "<span style='color:#b00'>Error, see the log below.</span>")
            with self.out:
                traceback.print_exc()
    return wrapper


class _Dropped(Exception):
    """Stops a background read that is no longer needed."""


class CropSession:
    def __init__(self, path="", name_filter="", out_folder="", dataset=None, box=None, lock_size=False,
                 skip_done=True, prefetch=True, copy_metadata=True, update_size_attrs=True,
                 workers=None, center=None, bf_radius=None, adf_inner=None, adf_outer=None,
                 contrast=(0.5, 99.5), gamma=1.0):
        self.out_folder, self.box, self.copy_metadata = out_folder, normalise_box(box), copy_metadata
        self.skip_done, self.update_size_attrs, self.workers = skip_done, update_size_attrs, workers
        self.geom_kw = dict(center=center, bf_radius=bf_radius, adf_inner=adf_inner, adf_outer=adf_outer)
        self._geom_kw0 = dict(self.geom_kw)  # the settings, for files opened before any detectors are applied
        self._custom = False                 # True once detectors were applied in the tool
        self._contrast0 = (float(contrast[0]), float(contrast[1])), float(gamma)
        self.dataset = dataset or None
        self.queue, self.root, self.qi = [], None, None
        self.cur = self.roi = self.prev_roi = self.sel = self.last_saved = self.last_geom = None
        self._requested = None
        self._failed = set()                 # files that couldn't be read
        self._busy = self._syncing = self._geom_hold = False
        self._drag = None                    # what is being dragged on the pattern panel
        self._pool = ThreadPoolExecutor(max_workers=1) if prefetch else None
        self._cache = {}                     # path -> Future of a background read
        self._build_ui(str(path or ""), name_filter or "", dataset or "", lock_size, center is None)

    # ---- where things go --------------------------------------------------------
    def out_path(self, p):
        if self.out_folder:
            base = Path(self.out_folder).expanduser()
            if self.root is not None and self.root in p.parents:     # mirror the subfolders
                base = base / p.parent.relative_to(self.root)
            return base / f"{p.stem}_crop.h5"
        return p.parent / "cropped" / f"{p.stem}_crop.h5"

    def summary_log(self):
        base = Path(self.out_folder).expanduser() if self.out_folder else self.root
        return base / "crop_log_all.csv"

    def is_done(self, j):
        return self.out_path(self.queue[j]).exists()

    def label(self, p):
        return str(p.relative_to(self.root)) if self.root is not None else p.name

    # ---- layout -----------------------------------------------------------------
    def _build_ui(self, path, name_filter, dataset, lock_size, recentre):
        with plt.ioff():
            self.fig, (self.ax, self.axd) = plt.subplots(
                1, 2, figsize=(10, 4.8), gridspec_kw=dict(width_ratios=[1.3, 1]))
        if not isinstance(self.fig.canvas, widgets.DOMWidget):
            plt.close(self.fig)
            raise RuntimeError("Needs the interactive widget backend: run `%matplotlib widget` in the first "
                               "cell (pip install ipympl, then restart the kernel).")
        self.fig.subplots_adjust(left=0.07, right=0.98, bottom=0.1, top=0.88, wspace=0.28)
        self.fig.canvas.header_visible = False
        self.fig.canvas.mpl_connect("key_press_event", self._on_key)
        self.fig.canvas.mpl_connect("button_press_event", self._on_dp_press)       # drag the detectors
        self.fig.canvas.mpl_connect("motion_notify_event", self._on_dp_motion)
        self.fig.canvas.mpl_connect("button_release_event", self._on_dp_release)

        L = widgets.Layout
        sty = {"description_width": "80px"}
        self.w_path = widgets.Text(value=path, description="file / folder", continuous_update=False,
                                   placeholder=r"a .h5 file, or a folder to go through all files in it and its subfolders",
                                   layout=L(width="600px"), style=sty)
        self.b_load = widgets.Button(description="Load", icon="folder-open", button_style="primary",
                                     layout=L(width="90px"))
        self.w_filter = widgets.Text(value=name_filter, description="name filter", continuous_update=False,
                                     placeholder="e.g. *reshaped*  (blank = every .h5/.hdf5)",
                                     layout=L(width="330px"), style=sty)
        self.w_dset = widgets.Text(value=dataset, description="dataset", continuous_update=False,
                                   placeholder="auto (largest 4D dataset)", layout=L(width="330px"), style=sty)
        self.w_view = widgets.ToggleButtons(options=VIEWS, description="Image", style={"button_width": "56px"})
        self.w_file = widgets.Dropdown(description="File", layout=L(width="520px"), style=sty)
        self.b_skip = widgets.Button(description="Skip", icon="forward", layout=L(width="90px"))
        self.w_count = widgets.HTML()
        self.queue_row = widgets.HBox([self.w_file, self.b_skip, self.w_count], layout=L(display="none"))
        self.w_title = widgets.HTML()

        def num(desc, lo):
            return widgets.BoundedIntText(value=lo, min=lo, max=100000, description=desc,
                                          layout=L(width="185px"), style={"description_width": "90px"})
        self.w_y0, self.w_x0 = num("axis 0 start", 0), num("axis 1 start", 0)
        self.w_h, self.w_w = num("axis 0 size", 1), num("axis 1 size", 1)
        self.w_lock = widgets.Checkbox(value=lock_size, description="lock size", indent=False,
                                       layout=L(width="100px"))
        self.w_info = widgets.HTML()
        self.b_centre = widgets.Button(description="Centre ROI", icon="crosshairs")
        self.b_crop = widgets.Button(description="Crop & save", icon="scissors",
                                     button_style="success", layout=L(width="190px"))
        self.w_prog = widgets.FloatProgress(value=0, min=0, max=1, layout=L(width="240px"))
        self.w_status = widgets.HTML()
        self.out = widgets.Output(layout=L(max_height="170px", overflow="auto",
                                           border="1px solid #ccc", padding="4px"))

        # display contrast (left, under the image): only changes what you see
        (lo, hi), gamma = self._contrast0
        self.w_clim = widgets.FloatRangeSlider(
            value=(lo, hi), min=0, max=100, step=0.1, readout_format=".1f", description="contrast %",
            layout=L(width="450px"), style=sty)
        self.w_gamma = widgets.FloatSlider(value=gamma, min=0.1, max=4, step=0.05, readout_format=".2f",
                                           description="gamma", layout=L(width="450px"), style=sty)
        self.b_creset = widgets.Button(description="Reset contrast", icon="undo", layout=L(width="150px"))
        self.w_cinfo = widgets.HTML()
        self.w_clim.observe(self._on_contrast, "value")
        self.w_gamma.observe(self._on_contrast, "value")
        self.b_creset.on_click(self._on_contrast_reset)

        # virtual detectors (right, under the diffraction pattern)
        gsty = {"description_width": "85px"}
        def gslider(desc, color):
            return widgets.FloatSlider(value=0, min=0, max=1, step=0.1, readout_format=".1f", description=desc,
                                       layout=L(width="400px"), style=dict(gsty, handle_color=color))
        self.w_cy, self.w_cx = gslider("centre 2", "#0bb"), gslider("centre 3", "#0bb")
        self.w_rbf = gslider("BF", "#0bb")
        self.w_rin, self.w_rout = gslider("ADF inner", "#3c3"), gslider("ADF outer", "#3c3")
        self.w_edge = widgets.Checkbox(value=True, description="to edge", indent=False, layout=L(width="80px"))
        self.b_apply = widgets.Button(description="Apply detectors", icon="refresh", layout=L(width="150px"),
                                      tooltip="recompute the virtual images with these detectors")
        self.b_find = widgets.Button(description="Find disk", icon="bullseye", layout=L(width="110px"),
                                     tooltip="put the detectors back where they are found automatically")
        self.w_recentre = widgets.Checkbox(value=recentre, description="find centre in each new file",
                                           indent=False, layout=L(width="220px"))
        self.w_ginfo = widgets.HTML()
        for w in (self.w_cy, self.w_cx, self.w_rbf, self.w_rin, self.w_rout, self.w_edge):
            w.observe(self._on_geom_fields, "value")
        self.b_apply.on_click(self._on_apply)
        self.b_find.on_click(self._on_find)
        self.w_recentre.observe(self._on_recentre, "value")

        box = dict(border="1px solid #ddd", padding="4px")
        display_panel = widgets.VBox([
            widgets.HTML("<b>Display</b> <span style='color:#777'>(only what you see; the data is "
                         "never changed)</span>"),
            self.w_clim, self.w_gamma, widgets.HBox([self.b_creset, self.w_cinfo])],
            layout=L(width="480px", **box))
        detector_panel = widgets.VBox([
            widgets.HTML("<b>Virtual detectors</b> <span style='color:#777'>(detector pixels; drag the "
                         "circles or the + on the pattern)</span>"),
            self.w_cy, self.w_cx, self.w_rbf, self.w_rin, widgets.HBox([self.w_rout, self.w_edge]),
            widgets.HBox([self.b_apply, self.b_find, self.w_recentre]), self.w_ginfo],
            layout=L(width="510px", **box))

        for w in (self.w_path, self.w_filter, self.w_dset):   # fire on Enter (or leaving the box)
            w.observe(self._on_inputs, "value")
        self.b_load.on_click(self._on_load_click)
        self.w_file.observe(self._on_file, "value")
        self.b_skip.on_click(self._on_skip)
        self.w_view.observe(self._on_view, "value")
        for w in (self.w_y0, self.w_x0, self.w_h, self.w_w):
            w.observe(self._on_fields, "value")
        self.b_centre.on_click(self._on_centre)
        self.b_crop.on_click(self._on_crop)

        self.ui = widgets.VBox([
            widgets.HBox([self.w_path, self.b_load]),
            widgets.HBox([self.w_filter, self.w_dset]),
            self.queue_row,
            self.w_title,
            self.w_view,
            self.fig.canvas,
            widgets.HBox([display_panel, detector_panel]),
            widgets.HBox([self.w_y0, self.w_x0, self.w_h, self.w_w, self.w_lock]),
            self.w_info,
            widgets.HBox([self.b_centre, self.b_crop]),
            widgets.HBox([self.w_prog, self.w_status]),
            self.out,
        ])

    def show(self):
        display(self.ui)
        if self.w_path.value.strip():
            self.request_load()
        else:
            self._clear_axes("Type the path of a file or folder above and press Enter")
            self._set_busy(False, "Waiting for a path.")

    # ---- choosing what to open -----------------------------------------------------
    @_safe
    def _on_inputs(self, _change):
        if not self._busy:
            self.request_load()

    @_safe
    def _on_load_click(self, _button):
        if not self._busy:          # reload only if the last attempt failed or an input changed
            self.request_load(force=self.cur is None)

    def request_load(self, force=False):
        text = self.w_path.value.strip()
        if not text:
            self._set_busy(False, "Type the path of a file or folder and press Enter.")
            return
        key = (text, self.w_filter.value.strip(), self.w_dset.value.strip())
        if key == self._requested and not force:
            return
        self._requested = key
        self._failed = set()
        self._drop_cache()
        self.dataset = key[2] or None
        path = clean_path(text)
        if path.is_dir():
            self._set_busy(True, f"Searching {path} ...")
            files = find_h5_files(path, key[1])
            if not files:
                self.queue, self.root = [], None
                self._refresh_queue_ui()
                return self._fail(f"No HDF5 files {'matching ' + repr(key[1]) + ' ' if key[1] else ''}"
                                  f"in {path} or its subfolders.")
            self.queue, self.root = files, path
            folders = len({p.parent for p in files})
            todo = self._todo_after(-1)
            with self.out:
                print(f"Found {len(files)} HDF5 file(s) in {folders} folder(s) under {path}; "
                      f"{len(files) - len(todo)} already cropped.")
            if not todo:
                with self.out:
                    print("Every file already has a crop. Pick one from the File list to redo it.")
            self.open_index(todo[0] if todo else 0, auto=bool(todo))
        elif path.is_file():
            self.queue, self.root = [path], None
            self.open_index(0)
        else:
            self.queue, self.root = [], None
            self._refresh_queue_ui()
            self._fail(f"Not found: {path}")

    def _todo_after(self, i):
        return [j for j in range(i + 1, len(self.queue))
                if self.queue[j] not in self._failed and not (self.skip_done and self.is_done(j))]

    def open_index(self, i, auto=False):
        """Open queue[i]. With auto=True, files that can't be read are logged and passed over."""
        while True:
            self.qi = i
            self._refresh_queue_ui()
            if self._load(self.queue[i]):
                self._prefetch_next()
                return
            nxt = self._todo_after(i)
            if not auto or not nxt:
                return
            i = nxt[0]

    @_safe
    def _on_file(self, change):
        if not self._syncing and not self._busy and change["new"] is not None:
            self.open_index(change["new"])

    @_safe
    def _on_skip(self, _button):
        if not self._busy:
            self._advance()

    def _advance(self):
        nxt = self._todo_after(self.qi)
        if nxt:
            return self.open_index(nxt[0], auto=True)
        left = sum(not self.is_done(j) and self.queue[j] not in self._failed for j in range(len(self.queue)))
        bad = f" ({len(self._failed)} file(s) couldn't be read, marked ✗)" if self._failed else ""
        msg = "Reached the end of the list. " + (
            f"{left} file(s) still have no crop; pick them from the File list." if left
            else f"<b>Every readable file has a crop.</b>") + bad
        self._set_busy(False, msg)
        with self.out:
            print(msg.replace("<b>", "").replace("</b>", ""))

    # ---- reading (in the foreground, or in the background for the next file) --------
    def _read(self, path, progress=None, geom_kw=None):
        with h5py.File(path, "r") as f:
            dset = find_4d_dataset(f, self.dataset)
            info = dict(path=path, dpath=dset.name, shape=dset.shape, dtype=dset.dtype, names=axis_names(dset))
            info["mean_dp"] = estimate_mean_dp(dset)          # from ~1000 patterns, for the detector panel
            info["geom"] = detector_geometry(dset, **(geom_kw or self.geom_kw), mean_dp=info["mean_dp"])
        info["vimgs"] = virtual_images(path, info["dpath"], info["geom"], self.workers, progress)
        return info

    def _prefetch_next(self):
        if self._pool is None or len(self.queue) < 2:
            return
        nxt = self._todo_after(self.qi)
        if not nxt:
            return
        p = self.queue[nxt[0]]
        for k in [k for k in self._cache if k != p]:      # keep only the next file
            self._stop(self._cache.pop(k))
        if p not in self._cache:
            state = {"frac": 0.0, "stop": False}

            def progress(frac, _text):
                if state["stop"]:                         # dropped: stop, so the workers are free again
                    raise _Dropped()
                state["frac"] = frac
            fut = self._pool.submit(self._read, p, progress, dict(self.geom_kw))
            fut.state = state
            self._cache[p] = fut

    @staticmethod
    def _stop(fut):
        fut.state["stop"] = True
        fut.cancel()

    def _drop_cache(self):
        for fut in self._cache.values():
            self._stop(fut)
        self._cache.clear()

    @_safe
    def _load(self, path):
        self.cur = None
        self.w_title.value = f"<h3 style='margin:4px 0'>{self.label(path)}</h3>"
        self._set_busy(True, f"Reading <b>{path.name}</b> ...")
        fut = self._cache.pop(path, None)
        try:
            if fut is not None and not fut.cancelled():
                while not fut.done():                     # already being read in the background
                    self._progress(fut.state["frac"], f"reading {path.name} (pre-loaded in the background) ...")
                    time.sleep(0.25)
                data = fut.result()
            else:
                data = self._read(path, progress=self._progress)
        except (OSError, KeyError, ValueError) as e:
            self._failed.add(path)
            self._refresh_queue_ui()
            self._fail(f"Could not read {path.name}: {e}")
            with self.out:
                print(f"✗ {self.label(path)}: {e}")
            return False
        self._failed.discard(path)
        data["out"] = self.out_path(path)
        self.cur = data
        self._set_title()
        self._draw()
        self._set_busy(False, "Draw or adjust the ROI, then <b>Crop &amp; save</b>.")
        return True

    def _set_title(self, extra=""):
        cur = self.cur
        pos = f"[{self.qi + 1}/{len(self.queue)}] " if len(self.queue) > 1 else ""
        warn = (" &nbsp; <b style='color:#a60'>(exists: saving overwrites it)</b>"
                if cur["out"].exists() and not extra else "")
        self.w_title.value = (
            f"<h3 style='margin:4px 0'>{pos}{self.label(cur['path'])}</h3>"
            f"<code>{cur['dpath']}</code> &nbsp; shape {cur['shape']} {cur['dtype']} &nbsp; "
            f"&rarr; saves to <code>{cur['out']}</code>{warn}{extra}")

    def _refresh_queue_ui(self):
        multi = len(self.queue) > 1
        self.queue_row.layout.display = "flex" if multi else "none"
        if not multi:
            return
        done = [self.is_done(j) for j in range(len(self.queue))]
        self._syncing = True
        try:
            mark = lambda p, d: "✓ " if d else ("✗ " if p in self._failed else "• ")
            self.w_file.options = [(mark(p, d) + self.label(p), j)
                                   for j, (p, d) in enumerate(zip(self.queue, done))]
            self.w_file.value = self.qi
        finally:
            self._syncing = False
        self.w_count.value = (f"&nbsp; <b>{sum(done)} / {len(self.queue) - len(self._failed)}</b> cropped"
                              + (f" &nbsp; ({len(self._failed)} unreadable)" if self._failed else ""))

    def _fail(self, msg):
        self._clear_axes(msg)
        self._set_busy(False, f"<span style='color:#b00'>{msg}</span>")

    def _clear_axes(self, text=""):
        self._drop_selector()
        for a in (self.ax, self.axd):
            a.clear()
            a.set_axis_off()
        self.ax.text(0.5, 0.5, text, ha="center", va="center", wrap=True, transform=self.ax.transAxes)
        self.fig.canvas.draw_idle()

    def _drop_selector(self):
        if self.sel is not None:
            self.sel.set_active(False)
            self.sel.disconnect_events()
            self.sel = None

    def _draw(self):
        cur = self.cur
        ny, nx = cur["shape"][:2]
        self._drop_selector()
        for a in (self.ax, self.axd):
            a.clear()
            a.set_axis_on()

        self._sync_views()
        img = cur["vimgs"][self.w_view.value]
        self.im = self.ax.imshow(img, cmap="gray", interpolation="nearest", norm=self._norm(img))
        n = cur["names"]                                    # the file's own axis names, as stored
        self.ax.set_ylabel(f"axis 0: {n[0]}")
        self.ax.set_xlabel(f"axis 1: {n[1]}")
        self.w_y0.description, self.w_x0.description = f"{n[0]} start", f"{n[1]} start"
        self.w_h.description, self.w_w.description = f"{n[0]} size", f"{n[1]} size"
        self.w_cy.description, self.w_cx.description = f"centre {n[2]}", f"centre {n[3]}"

        dp = cur["mean_dp"]
        ky, kx = cur["shape"][2:]
        self.axd.imshow(np.log1p(np.clip(dp - dp.min(), 0, None)), cmap="inferno", interpolation="nearest")
        self._c_bf = Circle((0, 0), 1, fill=False, ec="cyan", lw=1.2, label="BF")
        self._c_in = Circle((0, 0), 1, fill=False, ec="lime", lw=1.2, ls="--", label="ADF inner")
        self._c_out = Circle((0, 0), 1, fill=False, ec="lime", lw=1.2, ls=":", label="ADF outer")
        for c in (self._c_bf, self._c_in, self._c_out):
            self.axd.add_patch(c)
        self._c_mark, = self.axd.plot([], [], "+", color="cyan", ms=12, mew=1.5)
        self.axd.set_xlim(-0.5, kx - 0.5)                   # the circles must not widen the view
        self.axd.set_ylim(ky - 0.5, -0.5)
        self.axd.set_ylabel(f"axis 2: {n[2]}")
        self.axd.set_xlabel(f"axis 3: {n[3]}")
        self._set_geom_fields(cur["geom"])

        # state_modifier_keys: switch off matplotlib's "r" = rotate-the-box key, so the box you see is
        # always the axis-aligned area that gets cropped
        kw = dict(useblit=True, button=[1], minspanx=1, minspany=1, spancoords="data", interactive=True,
                  state_modifier_keys={"rotate": "no-key"})
        props = dict(edgecolor="yellow", fill=False, linewidth=1.8)
        try:
            self.sel = RectangleSelector(self.ax, self._on_select, props=props, **kw)
        except TypeError:                                   # matplotlib < 3.5
            self.sel = RectangleSelector(self.ax, self._on_select, rectprops=props, **kw)

        self._syncing = True
        try:
            self.w_h.max, self.w_w.max = ny, nx
            self.w_y0.max, self.w_x0.max = ny - 1, nx - 1
        finally:
            self._syncing = False

        roi = self.prev_roi                                 # carry over the last ROI if it fits
        if roi is None or roi[1] > ny or roi[3] > nx:
            roi = box_at(ny / 2, nx / 2, self._default_size(ny, nx), (ny, nx))
        self.apply_roi(roi)

    def _default_size(self, ny, nx):
        if self.w_lock.value and self.roi is not None:
            return self.roi[1] - self.roi[0], self.roi[3] - self.roi[2]
        return self.box or (max(1, ny // 3), max(1, nx // 3))

    # ---- ROI ------------------------------------------------------------------------
    def apply_roi(self, roi):
        ny, nx = self.cur["shape"][:2]
        y0, y1, x0, x1 = self.roi = clamp_roi(roi, (ny, nx))
        h, w = y1 - y0, x1 - x0
        self.sel.extents = (x0 - 0.5, x1 - 0.5, y0 - 0.5, y1 - 0.5)     # snapped to pixel edges
        self.sel.set_visible(True)
        self._syncing = True
        try:
            self.w_y0.value, self.w_x0.value, self.w_h.value, self.w_w.value = y0, x0, h, w
        finally:
            self._syncing = False
        new_shape = (h, w) + tuple(self.cur["shape"][2:])
        n = self.cur["names"]
        self.w_info.value = (f"ROI: {n[0]} <b>{y0}:{y1}</b>, {n[1]} <b>{x0}:{x1}</b> &rarr; <b>{h} &times; {w}</b> "
                             f"scan positions ({100 * h * w / (ny * nx):.1f}% of the scan) &nbsp; "
                             f"new shape {new_shape}, same axis order")
        self.ax.set_title(f"{self.w_view.value} virtual image   ROI {h} x {w} px", fontsize=10)
        self.fig.canvas.draw_idle()

    @_safe
    def _on_select(self, _press=None, _release=None):
        if self.cur is None:
            return
        xmin, xmax, ymin, ymax = self.sel.extents
        if self.w_lock.value:                      # keep the size, centre it where you drew
            size = (self.w_h.value, self.w_w.value)
            roi = box_at((ymin + ymax) / 2 + 0.5, (xmin + xmax) / 2 + 0.5, size, self.cur["shape"][:2])
        else:                                      # snap edges to the nearest pixel boundary
            edge = lambda v: int(np.floor(v + 1.0))
            roi = (edge(ymin), edge(ymax), edge(xmin), edge(xmax))
        self.apply_roi(roi)

    @_safe
    def _on_fields(self, _change):
        if self._syncing or self.cur is None:
            return
        ny, nx = self.cur["shape"][:2]
        h, w = min(max(self.w_h.value, 1), ny), min(max(self.w_w.value, 1), nx)
        y0, x0 = min(max(self.w_y0.value, 0), ny - h), min(max(self.w_x0.value, 0), nx - w)
        self.apply_roi((y0, y0 + h, x0, x0 + w))

    @_safe
    def _on_centre(self, _button):
        if self.cur is not None:
            ny, nx = self.cur["shape"][:2]
            y0, y1, x0, x1 = self.roi
            self.apply_roi(box_at(ny / 2, nx / 2, (y1 - y0, x1 - x0), (ny, nx)))

    def _on_key(self, event):
        if event.key == "escape" and self.cur is not None:    # the selector hides the box on Esc
            self.apply_roi(self.roi)

    @_safe
    def _on_view(self, change):
        if self._syncing or self.cur is None:
            return
        self._show_image()

    def _sync_views(self):
        names = [v for v in VIEWS if v in self.cur["vimgs"]]
        self._syncing = True
        try:
            keep = self.w_view.value
            self.w_view.options = names
            self.w_view.value = keep if keep in names else names[0]
        finally:
            self._syncing = False

    def _show_image(self):
        """Show the selected virtual image of the current file, keeping the ROI."""
        img = self.cur["vimgs"][self.w_view.value]
        self.im.set_data(img)
        self._apply_contrast(draw=False)
        self.apply_roi(self.roi)                           # redraws

    # ---- display contrast -------------------------------------------------------------------
    def _norm(self, img):
        return display_norm(img, self.w_clim.value, self.w_gamma.value)

    def _apply_contrast(self, draw=True):
        img = self.cur["vimgs"][self.w_view.value]
        norm = self._norm(img)
        self.im.norm = norm
        self.im.changed()
        self.w_cinfo.value = (f"&nbsp; black {norm.vmin:.4g}, white {norm.vmax:.4g}"
                              f"{'' if abs(self.w_gamma.value - 1) < 1e-3 else ', gamma %.2f' % self.w_gamma.value}")
        if draw:
            self.fig.canvas.draw_idle()

    @_safe
    def _on_contrast(self, _change):
        if self.cur is not None and not self._busy:
            self._apply_contrast()

    @_safe
    def _on_contrast_reset(self, _button):
        (lo, hi), gamma = self._contrast0
        self.w_clim.value, self.w_gamma.value = (lo, hi), gamma

    # ---- virtual detectors --------------------------------------------------------------------
    # The sliders (and dragging on the pattern) only move the circles; "Apply detectors" recomputes
    # the virtual images. Applied detectors are also used for the files opened after this one.
    def _geom_fields(self):
        return dict(cy=self.w_cy.value, cx=self.w_cx.value, r_bf=self.w_rbf.value, r_in=self.w_rin.value,
                    r_out=np.inf if self.w_edge.value else self.w_rout.value)

    def _set_geom_fields(self, g):
        ky, kx = self.cur["shape"][2:]
        rmax = float(np.ceil(np.hypot(ky, kx)))
        self._geom_hold = True
        try:
            self.w_cy.max, self.w_cx.max = ky - 1, kx - 1
            for w in (self.w_rbf, self.w_rin, self.w_rout):
                w.min, w.max = 0.5, rmax
            self.w_cy.value, self.w_cx.value = float(g["cy"]), float(g["cx"])
            self.w_rbf.value, self.w_rin.value = float(g["r_bf"]), float(g["r_in"])
            edge = not np.isfinite(g["r_out"])
            if edge:      # value used if "to edge" is unticked: the largest circle inside the detector
                inside = min(g["cx"] + 0.5, kx - 0.5 - g["cx"], g["cy"] + 0.5, ky - 0.5 - g["cy"])
                self.w_rout.value = round(max(inside, g["r_in"] + 1), 1)
            else:
                self.w_rout.value = float(g["r_out"])
            self.w_edge.value = edge
        finally:
            self._geom_hold = False
        self._show_geom()

    def _show_geom(self, draw=True):
        """Move the circles to the slider values and say whether they still need applying."""
        g = self._geom_fields()
        c = (g["cx"], g["cy"])
        for circ, r in ((self._c_bf, g["r_bf"]), (self._c_in, g["r_in"]), (self._c_out, g["r_out"])):
            circ.set_visible(bool(np.isfinite(r)))
            circ.center = c
            if np.isfinite(r):
                circ.set_radius(r)
        self._c_mark.set_data([g["cx"]], [g["cy"]])
        self.axd.legend(handles=[h for h in (self._c_bf, self._c_in, self._c_out) if h.get_visible()],
                        loc="upper right", fontsize=8, framealpha=0.6)
        pending = not same_geom(g, self.cur["geom"])
        self.axd.set_title(f"Mean diffraction pattern (log)\ncentre ({g['cy']:.1f}, {g['cx']:.1f})   "
                           f"BF {g['r_bf']:.1f}   ADF {g['r_in']:.1f} to {fmt_r(g['r_out'])}"
                           + ("   (not applied)" if pending else ""),
                           fontsize=9, color="#c60" if pending else "black")
        self.w_rout.disabled = self._busy or self.w_edge.value
        self.b_apply.button_style = "warning" if pending else ""
        if pending:
            bad = np.isfinite(g["r_out"]) and g["r_out"] <= g["r_in"]
            self.w_ginfo.value = ("<span style='color:#b00'>ADF outer must be larger than ADF inner.</span>" if bad
                                  else "<span style='color:#c60'>Changed: press <b>Apply detectors</b> to "
                                       "recompute the images.</span>")
        else:
            self.w_ginfo.value = self._next_geom_text()
        if draw:
            self.fig.canvas.draw_idle()

    def _next_geom_text(self):
        kw = self.geom_kw
        if kw == self._geom_kw0 and all(v is None for v in kw.values()):
            return "<span style='color:#777'>Next files: detectors found automatically in each file.</span>"
        f = lambda v, auto: auto if v is None else f"{v:.1f}"
        centre = "found in each file" if kw["center"] is None else "(%.1f, %.1f)" % tuple(kw["center"])
        txt = (f"centre {centre}, BF {f(kw['bf_radius'], 'found in each file')}, "
               f"ADF {f(kw['adf_inner'], '1.2 x BF')} to {f(kw['adf_outer'], 'edge')}")
        return f"<span style='color:#777'>Next files: {txt}.</span>"

    @_safe
    def _on_geom_fields(self, _change):
        if not self._geom_hold and self.cur is not None:
            self._show_geom()

    def _update_next_geom(self, g):
        """Detectors for files opened from now on (not the current one). Returns True if they changed."""
        if self._custom:                                   # applied in the tool: keep the radii
            kw = dict(bf_radius=float(g["r_bf"]), adf_inner=float(g["r_in"]),
                      adf_outer=float(g["r_out"]) if np.isfinite(g["r_out"]) else None)
        else:
            kw = dict(self._geom_kw0)
        kw["center"] = None if self.w_recentre.value else (float(g["cy"]), float(g["cx"]))
        if kw == self.geom_kw:
            return False
        self.geom_kw = kw
        self._drop_cache()                                 # a pre-read next file used the old detectors
        return True

    @_safe
    def _on_apply(self, _button):
        if self._busy or self.cur is None:
            return
        g, cur = self._geom_fields(), self.cur
        if np.isfinite(g["r_out"]) and g["r_out"] <= g["r_in"]:
            return self._show_geom()
        self._custom = True
        self._update_next_geom(g)
        if not same_geom(g, cur["geom"]):
            self._set_busy(True, "Recomputing the virtual images with the new detectors ...")
            vimgs = virtual_images(cur["path"], cur["dpath"], g, self.workers, progress=self._progress)
            cur["geom"], cur["vimgs"] = g, vimgs
            self._sync_views()
            self._show_image()
            msg = "Detectors applied. Draw or adjust the ROI, then <b>Crop &amp; save</b>."
        else:
            msg = "Detectors unchanged; the next files will use them too."
        self._show_geom()
        self._set_busy(False, msg)
        self._prefetch_next()

    @_safe
    def _on_find(self, _button):
        """Back to the automatically found detectors (and the settings) for this file. Not applied yet."""
        if self.cur is not None and not self._busy:
            kw = dict(self._geom_kw0)
            if self.w_recentre.value:
                kw["center"] = None
            g = detector_geometry(None, **kw, mean_dp=self.cur["mean_dp"])
            self._set_geom_fields(g)

    @_safe
    def _on_recentre(self, _change):
        if self.cur is not None and not self._busy:
            if self._update_next_geom(self.cur["geom"]):
                self._prefetch_next()
            self._show_geom()

    # dragging on the pattern panel: grab the + (centre) or a circle (radius); clicking anywhere
    # else moves the centre there
    @_safe
    def _on_dp_press(self, ev):
        if (ev.inaxes is not self.axd or ev.button != 1 or ev.xdata is None or self.cur is None
                or self._busy or self.axd.get_navigate_mode() is not None):
            return
        g = self._geom_fields()
        d = np.hypot(ev.xdata - g["cx"], ev.ydata - g["cy"])
        tol = max(3.0, 0.03 * max(self.cur["shape"][2:]))
        near = [(d, "centre")] + [(abs(d - g[k]), k) for k in ("r_bf", "r_in", "r_out") if np.isfinite(g[k])]
        dist, what = min(near)
        if dist > tol:                                     # not on a handle: centre jumps here
            self._drag = ("centre", (0.0, 0.0))
            self._on_dp_motion(ev)
        elif what == "centre":
            self._drag = ("centre", (g["cx"] - ev.xdata, g["cy"] - ev.ydata))
        else:
            self._drag = (what, g[what] - d)

    @_safe
    def _on_dp_motion(self, ev):
        if self._drag is None or ev.inaxes is not self.axd or ev.xdata is None:
            return
        what, off = self._drag
        self._geom_hold = True
        try:
            if what == "centre":
                self.w_cx.value, self.w_cy.value = round(ev.xdata + off[0], 1), round(ev.ydata + off[1], 1)
            else:
                g = self._geom_fields()
                r = np.hypot(ev.xdata - g["cx"], ev.ydata - g["cy"]) + off
                {"r_bf": self.w_rbf, "r_in": self.w_rin, "r_out": self.w_rout}[what].value = round(max(r, 0.5), 1)
        finally:
            self._geom_hold = False
        self._show_geom()

    def _on_dp_release(self, _ev):
        self._drag = None

    # ---- saving ---------------------------------------------------------------------------
    @_safe
    def _on_crop(self, _button):
        if self._busy or self.cur is None:
            return
        cur, roi, view = self.cur, self.roi, self.w_view.value
        out = cur["out"]
        out.parent.mkdir(parents=True, exist_ok=True)
        y0, y1, x0, x1 = roi
        self._set_busy(True, f"Writing <b>{out.name}</b> ...")
        notes = write_crop(cur["path"], cur["dpath"], roi, out, self.copy_metadata,
                           self.update_size_attrs, progress=self._progress)
        back = verify_crop(cur["path"], cur["dpath"], roi, out, cur["vimgs"]["Sum"], self.workers,
                           progress=self._progress)
        n = cur["names"]
        save_preview(cur["vimgs"][view], roi, back, out.parent / f"{cur['path'].stem}_roi.png",
                     f"{self.label(cur['path'])} ({view})", n, self.w_clim.value, self.w_gamma.value)
        row = dict(file=str(cur["path"]), dataset=cur["dpath"], axis0_name=n[0], axis1_name=n[1],
                   original_shape=str(tuple(cur["shape"])), axis0_start=y0, axis0_stop=y1,
                   axis1_start=x0, axis1_stop=x1, new_shape=str((y1 - y0, x1 - x0) + tuple(cur["shape"][2:])),
                   verified="identical, same axis order", output=str(out))
        append_log(out.parent / "crop_log.csv", row)
        if self.root is not None:
            self.summary_log().parent.mkdir(parents=True, exist_ok=True)
            append_log(self.summary_log(), row)
        with self.out:
            print(f"✓ {self.label(cur['path'])}: {n[0]} {y0}:{y1}, {n[1]} {x0}:{x1} -> "
                  f"{(y1 - y0, x1 - x0) + tuple(cur['shape'][2:])}  saved {out}")
            print("    checked: read back identical to the original, axis order unchanged")
            for n in notes:
                print("    note:", n)
        self.prev_roi, self.last_saved, self.last_geom = roi, out, dict(cur["geom"])
        self._set_title(" &nbsp; <b style='color:#080'>✓ saved</b>")
        self._refresh_queue_ui()
        if len(self.queue) > 1:
            self._set_busy(False)
            self._advance()
        else:
            self._set_busy(False, f"Saved <b>{out.name}</b>. Type the next path above.")

    # ---- status -------------------------------------------------------------------------
    def _set_busy(self, busy, msg=None):
        self._busy = busy
        for w in (self.w_path, self.w_filter, self.w_dset, self.b_load, self.w_file, self.b_skip):
            w.disabled = busy
        for w in (self.b_crop, self.b_centre, self.w_view, self.w_y0, self.w_x0, self.w_h, self.w_w,
                  self.w_lock, self.w_clim, self.w_gamma, self.b_creset, self.w_cy, self.w_cx, self.w_rbf,
                  self.w_rin, self.w_edge, self.b_apply, self.b_find, self.w_recentre):
            w.disabled = busy or self.cur is None
        self.w_rout.disabled = busy or self.cur is None or self.w_edge.value
        if msg is not None:
            self.w_status.value = msg

    def _progress(self, frac, text):
        self.w_prog.value = frac
        self.w_status.value = text

## 4. Crop

Run the cell below, then:

- **Type or paste a path** into *file / folder* and press **Enter** (or click **Load**). Quotes
  around the path are fine.
  - A **file** opens on its own.
  - A **folder** (e.g. the one holding your 38 data folders) lines up every HDF5 file inside it and
    its subfolders. Use **name filter** if other HDF5 files live there too (e.g. `*reshaped*`).
    Files that already have a crop are skipped, so you can stop and carry on later. Use the
    **File** list to jump to any file (✓ = cropped) and **Skip** to pass one over. Files without 4D
    data are passed over automatically and listed in the log.
- Leave **dataset** blank to use the largest 4D dataset in each file, or type its internal path (e.g. `/data`).
- **Draw** the ROI by dragging on the left image; drag inside the box to move it and drag its
  handles to resize it. You can also type the **start** and **size** along each scan axis (press Enter);
  the boxes are named after your file's axes (e.g. `scan_x start`, `scan_y size`).
- Tick **lock size** to keep the size fixed: wherever you drag, a box of that size is placed centred
  there (set `BOX` in the settings to start from a particular size, e.g. `BOX = 64`).
- **Image** switches between ADF, BF and total-sum virtual images.
- **Display** (under the image) sets the contrast: the **contrast %** range picks which percentiles of
  the image show as black and white (narrow it to bring out faint detail), and **gamma** below 1
  brightens faint areas, above 1 darkens them. It changes only what you see, applies straight away, is
  kept when you switch image or move to the next file, and is used for the saved `<name>_roi.png`.
  **Reset contrast** goes back to `CONTRAST` / `GAMMA` from the settings.
- **Virtual detectors** (under the pattern) set the BF disk and ADF annulus, in detector pixels. Use
  the sliders (the numbers can be typed), or drag on the pattern: grab the **+** to move the centre,
  grab a circle to resize it, or click anywhere else to put the centre there. The circles move
  straight away; press **Apply detectors** (orange while there are changes) to recompute the images
  (as long as opening the file takes; the ROI is kept). Untick **to edge** to give the ADF an outer
  radius. **Find disk** puts the detectors back where they're found automatically.
- Applied detectors are also used for the **next files**: the radii stay fixed and, with **find centre
  in each new file** ticked, the centre is found again in each file (untick it to keep this centre too).
  The line under the buttons says what the next files will use. Until you apply anything, each file
  gets the detectors from the settings (`CENTER`, `BF_RADIUS`, ...), found automatically where `None`.
- **Orientation:** the image is the data exactly as stored: axis 0 runs down, axis 1 runs across,
  both labelled with the names from your file. Nothing is transposed, flipped or rotated, on screen or
  in the saved file. After saving, the file is read back and compared with the original; the
  `<name>_roi.png` shows the original with the box next to the saved data, in the same orientation.
- **Crop & save** writes `<name>_crop.h5` (same internal dataset path, metadata copied),
  `<name>_roi.png` and a line in `crop_log.csv` to a `cropped` folder next to each file (or under
  `OUT_FOLDER`), and moves on to the next file. With a folder, `crop_log_all.csv` in the top folder
  lists every crop. Your last ROI is used as the starting point for the next file.
- While you draw, the next file is already being read in the background, so it usually opens at once.
- To zoom in for precise placement, use the magnifier in the plot toolbar; click it again to go
  back to drawing.

In [6]:
session = CropSession(PATH, NAME_FILTER, OUT_FOLDER, dataset=DATASET, box=BOX, lock_size=LOCK_SIZE,
                      skip_done=SKIP_DONE, prefetch=PREFETCH, copy_metadata=COPY_METADATA,
                      update_size_attrs=UPDATE_SIZE_ATTRS, workers=WORKERS,
                      center=CENTER, bf_radius=BF_RADIUS, adf_inner=ADF_INNER, adf_outer=ADF_OUTER,
                      contrast=CONTRAST, gamma=GAMMA)
session.show()

## 5. Check the result (optional)
Shows the virtual images of the file you saved last. To look at another one, set `path` yourself.

In [ ]:
path = session.last_saved        # or e.g. Path(r"D:\4DSTEM\cropped\scan_01_crop.h5")
if path is None:
    print("Nothing saved yet.")
else:
    log_path = session.summary_log() if session.root is not None else path.parent / "crop_log.csv"
    if log_path.exists():
        print(log_path.read_text())
    with h5py.File(path, "r") as f:
        d = find_4d_dataset(f, DATASET)
        names = axis_names(d)
        print(path.name, d.name, d.shape, d.dtype, "| axes:", names)
        print("cropped from", d.attrs.get("crop_source_file"), d.attrs.get("crop_original_shape"),
              f"| {names[0]} {d.attrs.get('crop_axis0_range')}, {names[1]} {d.attrs.get('crop_axis1_range')}")
        if path == session.last_saved and session.last_geom is not None:
            geom = session.last_geom                   # the detectors used in the tool for this file
        else:
            geom = detector_geometry(d, CENTER, BF_RADIUS, ADF_INNER, ADF_OUTER)
        dpath = d.name
    vimgs = virtual_images(path, dpath, geom, WORKERS)
    fig, axs = plt.subplots(1, len(vimgs), figsize=(3.3 * len(vimgs), 3.4))
    for a, (name, img) in zip(np.atleast_1d(axs), vimgs.items()):
        a.imshow(img, cmap="gray", norm=display_norm(img, session.w_clim.value, session.w_gamma.value))
        a.set_title(f"{name} ({img.shape[0]} x {img.shape[1]})", fontsize=10)
        a.set_ylabel(f"axis 0: {names[0]}", fontsize=8)
        a.set_xlabel(f"axis 1: {names[1]}", fontsize=8)
    fig.suptitle(path.name, fontsize=10)
    fig.tight_layout()